In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from sklearn.metrics import (
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import wandb

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("frozen_mlp_MCI.ipynb"))
os.chdir(notebook_dir)

# -------------------
# Class balance (kept from your code)
# -------------------
_NUM_POS = 0.29
_NUM_NEG = 0.71
_POS_WEIGHT = torch.tensor([_NUM_NEG / _NUM_POS])  # e.g., ~2.45; keep your value

# -------------------
# Small cosine schedule helper (0-indexed epochs)
# -------------------
def cosine_schedule(start_val, end_val, total_epochs, warmup_epochs=0):
    def schedule(epoch):
        if epoch < warmup_epochs:
            return start_val + (end_val - start_val) * epoch / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        return end_val + (start_val - end_val) * cosine_decay
    return schedule

# -------------------
# Model
# -------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)  # logits
        )
    def forward(self, x):
        return self.net(x)

# -------------------
# Utilities
# -------------------
def build_Xy(ids, features_dict, class_map):
    """
    Filters subjects to those present in both `features_dict` and `class_map`,
    preserving the order in `ids`. Returns X, y, subj_ids_kept.
    """
    kept_ids = [sid for sid in ids if sid.split('/')[0] in class_map and sid in features_dict]
    if len(kept_ids) == 0:
        raise ValueError("No overlapping subjects found between ids, features, and labels.")
    X = np.vstack([features_dict[sid] for sid in kept_ids])
    y = np.array([class_map[sid.split('/')[0]] for sid in kept_ids], dtype=np.int64)
    return X, y, kept_ids

def normalize_like_train(X, train_mean, train_std):
    return (X - train_mean) / (train_std + 1e-8)

def save_predictions_csv(csv_path, subject_ids, y_true, y_score, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(int),
        "y_score": y_score.astype(float),
        "y_pred": y_pred.astype(int),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

@torch.no_grad()
def evaluate_model(model, X, y, subject_ids, batch_size=64, device="cpu"):
    """
    Runs forward passes, computes metrics, and returns:
    metrics_dict, y_true (np), y_score (np), y_pred (np)
    """
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32).unsqueeze(1))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    scores, preds_bin, y_true = [], [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        scores.append(probs)
        y_true.append(yb.cpu().numpy().flatten())

    y_score = np.concatenate(scores)
    y_true = np.concatenate(y_true).astype(int)
    y_pred = (y_score > 0.5).astype(int)

    metrics = {
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "auprc": average_precision_score(y_true, y_score),
        "auc": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else np.nan,
    }
    return metrics, y_true, y_score, y_pred

# -------------------
# Training loop with best-checkpoint saving
# -------------------
def train_model(
    X_train, y_train, train_ids_kept,
    X_val, y_val, val_ids_kept,
    input_dim, feat_file, project_name,
    epochs=50, batch_size=32, base_lr=1e-5,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.float32).unsqueeze(1))
    val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                torch.tensor(y_val, dtype=torch.float32).unsqueeze(1))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, drop_last=False)

    model = MLP(input_dim=input_dim).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=base_lr,
        betas=(0.9, 0.999),
        weight_decay=0.04
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=_POS_WEIGHT.to(device))

    wandb.init(project=project_name, reinit=True)
    wandb.config.update({"epochs": epochs, "batch_size": batch_size, "base_lr": base_lr})

    lr_schedule = cosine_schedule(start_val=base_lr, end_val=1e-6, total_epochs=epochs, warmup_epochs=0)
    wd_schedule = cosine_schedule(start_val=0.04,    end_val=0.01, total_epochs=epochs, warmup_epochs=0)

    # Best-checkpoint tracking
    best_val = -float("inf")
    best_epoch = -1
    best_ckpt_path_state = None
    best_ckpt_path_full  = None
    key_val = 'bal_acc'  # choose validation metric to maximize

    run_name = wandb.run.name if wandb.run is not None else "unnamed_run"
    save_dir = wandb.run.dir if wandb.run is not None else "checkpoints"
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):
        # update LR/WD
        lr = lr_schedule(epoch)
        wd = wd_schedule(epoch)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
            pg['weight_decay'] = wd

        # -------- Train --------
        model.train()
        train_loss_sum = 0.0
        n_train = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            bs = xb.size(0)
            train_loss_sum += loss.item() * bs
            n_train += bs

        train_loss = train_loss_sum / max(1, n_train)

        # -------- Validate --------
        model.eval()
        val_loss_sum = 0.0
        n_val = 0
        val_scores, val_truth = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                bs = xb.size(0)
                val_loss_sum += loss.item() * bs
                n_val += bs
                probs = torch.sigmoid(logits).cpu().numpy().flatten()
                val_scores.append(probs)
                val_truth.append(yb.cpu().numpy().flatten())

        val_loss = val_loss_sum / max(1, n_val)
        y_score_val = np.concatenate(val_scores)
        y_true_val  = np.concatenate(val_truth).astype(int)
        y_pred_val  = (y_score_val > 0.5).astype(int)

        bal_acc = balanced_accuracy_score(y_true_val, y_pred_val)
        auprc   = average_precision_score(y_true_val, y_score_val)
        auc     = roc_auc_score(y_true_val, y_score_val) if len(np.unique(y_true_val)) > 1 else np.nan

        wandb.log({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "bal_acc": bal_acc,
            "auprc": auprc,
            "auc": auc,
            "lr": lr,
            "weight_decay": wd
        })

        # ---- Save best checkpoint by key_val ----
        current_key_val = bal_acc  # key_val is 'bal_acc'
        if current_key_val > best_val:
            best_val = current_key_val
            best_epoch = epoch + 1

            # Save both state_dict and full model
            best_ckpt_path_state = os.path.join(save_dir, f"{run_name}_BEST_state_dict.pth")
            best_ckpt_path_full  = os.path.join(save_dir, f"{run_name}_BEST_full_model.pth")
            torch.save(model.state_dict(), best_ckpt_path_state)
            torch.save(model, best_ckpt_path_full)

            # Also save the corresponding validation predictions CSV
            csv_path_val = os.path.join(save_dir, f"{run_name}_BEST_val_predictions.csv")
            save_predictions_csv(csv_path_val, val_ids_kept, y_true_val, y_score_val, y_pred_val)
    run_code = wandb.run.id if wandb.run is not None else "no_wandb"
    wandb.summary["best_epoch"] = best_epoch
    wandb.summary["best_val"] = best_val
    wandb.finish()

    return {
        "best_epoch": best_epoch,
        "best_val": best_val,
        "best_state_dict": best_ckpt_path_state,
        "best_full_model": best_ckpt_path_full,
        "save_dir": save_dir,
        "run_name": run_code
    }

In [ ]:
# ---- Paths / Inputs ----
summary_rows = []
for split_num in range(5):
    _NUM_RUNS = 10
    best_val_fold = -float('inf')
    best_run_info = None

    for ii in range(_NUM_RUNS):

        train_ids = np.loadtxt(f"../../splits/adni/train_subject_list_adni_{split_num}", dtype=str)
        val_ids   = np.loadtxt(f"../../splits/adni/val_subject_list_adni_{split_num}", dtype=str)
        test_ids  = np.loadtxt(f"../../splits/adni/test_subject_list_adni_{split_num}", dtype=str)  # NEW: test list

        df = pd.read_csv("../../metadata/adni_metadata.csv")

        # Map research group -> binary label
        class_map = dict(zip(df["subject_id"], df["entry_research_group"]))
        class_map = {k: (1 if v == "MCI" else 0) for k, v in class_map.items()}

        # Features
        feat_file = "../../latents/cls_adni_k8pcq4ai_300.npz"
        features_dict = np.load(feat_file, allow_pickle=True)

        # ---- Build filtered X/y with aligned subject IDs ----
        X_train_raw, y_train, train_ids_kept = build_Xy(train_ids, features_dict, class_map)
        X_val_raw,   y_val,   val_ids_kept   = build_Xy(val_ids,   features_dict, class_map)
        X_test_raw,  y_test,  test_ids_kept  = build_Xy(test_ids,  features_dict, class_map)

        X_train_raw[np.isnan(X_train_raw)]=0
        X_val_raw[np.isnan(X_val_raw)]=0
        X_test_raw[np.isnan(X_test_raw)]=0

        # ---- Normalize (train statistics only) ----
        X_mean = X_train_raw.mean(axis=0, keepdims=True)
        X_std  = X_train_raw.std(axis=0, keepdims=True) + 1e-8

        X_train = normalize_like_train(X_train_raw, X_mean, X_std)
        X_val   = normalize_like_train(X_val_raw,   X_mean, X_std)
        X_test  = normalize_like_train(X_test_raw,  X_mean, X_std)

        # ---- Train and track best checkpoint on validation ----
        train_out = train_model(
            X_train, y_train, train_ids_kept,
            X_val,   y_val,   val_ids_kept,
            input_dim=X_train.shape[1],
            feat_file=feat_file,
            project_name=f"prediction_mlp_mci_frozen_cv_{split_num}",
            epochs=50,
            batch_size=32,
            base_lr=5e-5
        )

        # ---- Load best model and evaluate on TEST (save CSV, do NOT show) ----
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        best_state_path = train_out["best_state_dict"]
        save_dir = train_out["save_dir"]
        run_name = train_out["run_name"]

        # Recreate model and load best weights
        best_model = MLP(input_dim=X_train.shape[1]).to(device)
        if best_state_path is None or not os.path.exists(best_state_path):
            raise FileNotFoundError("Best state_dict checkpoint was not saved or path not found.")
        best_model.load_state_dict(torch.load(best_state_path, map_location=device))

        # Evaluate on test
        val_metrics, __, __, __  = evaluate_model(
            best_model, X_val, y_val, val_ids_kept, batch_size=64, device=device
        )

        test_metrics, y_true_t, y_score_t, y_pred_t = evaluate_model(
            best_model, X_test, y_test, test_ids_kept, batch_size=64, device=device
        )

        # Save CSV (no plotting)
        csv_path_test = os.path.join(save_dir, f"{run_name}_TEST_predictions.csv")
        save_predictions_csv(csv_path_test, test_ids_kept, y_true_t, y_score_t, y_pred_t)

        # Optional: print a brief summary to console
        print(f"[Best @ epoch {train_out['best_epoch']}] val {train_out['best_val']:.4f}")
        #print(f"TEST bal_acc={test_metrics['bal_acc']:.4f} | AUPRC={test_metrics['auprc']:.4f} | AUC={test_metrics['auc']:.4f}")
        print(f"Saved VAL preds: {os.path.join(save_dir, f'{run_name}_BEST_val_predictions.csv')}")
        print(f"Saved TEST preds: {csv_path_test}")

        txt_path = os.path.join(save_dir, f"{run_name}_TEST_summary.txt")
        with open(txt_path, "w") as f:
            f.write(f"[Best @ epoch {train_out['best_epoch']}] val {train_out['best_val']:.4f}\n")
            f.write(f"TEST bal_acc={test_metrics['bal_acc']:.4f} | "
                    f"AUPRC={test_metrics['auprc']:.4f} | "
                    f"AUC={test_metrics['auc']:.4f}\n")
            f.write(f"VAL predictions CSV: {os.path.join(save_dir, f'{run_name}_BEST_val_predictions.csv')}\n")
            f.write(f"TEST predictions CSV: {csv_path_test}\n")

        # Track best run for this fold
        if train_out['best_val'] > best_val_fold:
            best_val_fold = train_out['best_val']
            best_run_info = {
                'split': split_num,
                'run_name': run_name,
                'best_val': train_out['best_val'],
                'best_epoch': train_out['best_epoch'],
                'bal_acc': val_metrics['bal_acc'],
                'auprc': val_metrics['auprc'],
                'auc': val_metrics['auc'],
            }

    # After all runs for this fold, append best run info
    summary_rows.append(best_run_info)

# Show summary table
import pandas as pd
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

